# sample5 - RAG + FAISS

**RAG（Retrieval-Augmented Generation）** は独自ドキュメントをベクトルDBに格納し、  
LLM の回答精度を向上させる技術です。

```
質問
 ↓
ベクトルDB（FAISS）で関連ドキュメントを検索
 ↓
関連ドキュメント + 質問 を Ollama（Docker）に渡す
 ↓
回答
```

| ステップ | 内容 |
|----------|------|
| 0 | Docker コンテナの確認 |
| 1 | ドキュメントの準備 |
| 2 | 埋め込みベクトルに変換 |
| 3 | FAISS インデックスの構築 |
| 4 | 類似ドキュメントの検索 |
| 5 | Ollama（Docker）と組み合わせた RAG |

## 事前準備

```bash
docker ps | grep ollama   # コンテナ起動確認
docker start ollama       # 停止中の場合
```

## 0. Docker コンテナの確認

In [1]:
import subprocess
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import ollama

result = subprocess.run(
    ['docker', 'ps', '--filter', 'name=ollama', '--format', '{{.Names}}\t{{.Status}}'],
    capture_output=True, text=True
)
print("Ollama コンテナ:", result.stdout.strip() if result.stdout else '未起動')
print("FAISS バージョン:", faiss.__version__)

Ollama コンテナ: ollama	Up About an hour
FAISS バージョン: 1.13.2


## 1. ドキュメントの準備

In [2]:
documents = [
    "PyTorch は Meta が開発したディープラーニングフレームワークです。動的計算グラフが特徴で、研究分野で広く使われています。",
    "TensorFlow は Google が開発したディープラーニングフレームワークです。Keras を高レベル API として使います。",
    "scikit-learn は Python の機械学習ライブラリで、分類・回帰・クラスタリングなどの古典的なアルゴリズムを提供します。",
    "RAG（検索拡張生成）は、LLM の回答精度を向上させるために外部ドキュメントを参照する技術です。",
    "FAISS は Meta が開発した高速ベクトル検索ライブラリです。大規模な埋め込みベクトルの類似検索に使います。",
    "LangChain は LLM を使ったアプリケーション開発フレームワークです。チェーン・エージェント・RAG 構築に使います。",
    "LoRA は大規模モデルを少ないパラメータで効率よく Fine-tuning する手法です。",
    "Ollama は Docker コンテナで動かすことで WSL 環境を汚さずにローカル LLM を実行できます。",
]
print(f"ドキュメント数: {len(documents)}")

ドキュメント数: 8


## 2. 埋め込みベクトルに変換

In [3]:
embed_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings  = embed_model.encode(documents, show_progress_bar=True).astype('float32')
print(f"埋め込み形状: {embeddings.shape}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

埋め込み形状: (8, 384)


## 3. FAISS インデックスの構築

In [4]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"登録ベクトル数: {index.ntotal}")

登録ベクトル数: 8


## 4. 類似ドキュメントの検索

In [5]:
def search(query, k=3):
    query_vec = embed_model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k)
    return [{'document': documents[i], 'distance': d}
            for d, i in zip(distances[0], indices[0])]

for query in ['Googleが開発した機械学習ツールは？', 'LLM の知識を補完する技術は？']:
    print(f"クエリ: {query}")
    for i, r in enumerate(search(query, k=2)):
        print(f"  [{i+1}] (距離:{r['distance']:.4f}) {r['document']}")
    print()

クエリ: Googleが開発した機械学習ツールは？
  [1] (距離:15.7677) TensorFlow は Google が開発したディープラーニングフレームワークです。Keras を高レベル API として使います。
  [2] (距離:18.3128) RAG（検索拡張生成）は、LLM の回答精度を向上させるために外部ドキュメントを参照する技術です。

クエリ: LLM の知識を補完する技術は？
  [1] (距離:11.6207) RAG（検索拡張生成）は、LLM の回答精度を向上させるために外部ドキュメントを参照する技術です。
  [2] (距離:13.5770) LangChain は LLM を使ったアプリケーション開発フレームワークです。チェーン・エージェント・RAG 構築に使います。



## 5. Ollama（Docker）と組み合わせた RAG

In [6]:
def rag_answer(question, k=3):
    context = "\n".join([r['document'] for r in search(question, k=k)])
    prompt  = f"""以下の情報をもとに質問に答えてください。

【参照情報】
{context}

【質問】
{question}

【回答】"""
    # Ollama は Docker コンテナ内で動作。localhost:11434 経由でアクセス
    response = ollama.chat(
        model='llama3',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content']

for q in ['PyTorch と TensorFlow の違いを教えてください。', 'FAISS とは何ですか？']:
    print(f"質問: {q}")
    print(f"回答: {rag_answer(q)}")
    print()

質問: PyTorch と TensorFlow の違いを教えてください。
回答: Here's the answer:

PyTorch and TensorFlow are both popular deep learning frameworks, but they have some key differences.

One major difference is that PyTorch uses a dynamic computation graph, which means that the graph is built on-the-fly as the model is being trained or used. This allows for more flexibility and ease of use, especially when working with complex models. In contrast, TensorFlow uses a static computation graph, where the entire graph is defined upfront before training or inference.

Another difference is in their approach to automatic differentiation (backpropagation). PyTorch uses just-in-time (JIT) compilation to optimize the computation graph for performance, while TensorFlow uses a more traditional approach of explicitly defining the computation graph and then using that to compute gradients.

Additionally, PyTorch has a stronger focus on dynamic neural networks and reinforcement learning, while TensorFlow is more geared to

# ダウンロードしたモデルを削除

In [1]:
import shutil
import os

hf_cache = os.path.expanduser('~/.cache/huggingface/hub')

models_to_delete = [
    'models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2',
]

for model_dir in models_to_delete:
    path = os.path.join(hf_cache, model_dir)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"削除: {model_dir}")
    else:
        print(f"なし: {model_dir}")

なし: models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2
